# Random haplotypes

Code for the analysis of randomly shared haplotypes, for calculating uncertainty in TP53 haplotype calling

In [1]:
import os
import networkx as nx
import subprocess
import random
from pathlib import Path
import scipy.stats as stats

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Rectangle
import itertools

import matplotlib
matplotlib.rc_file_defaults()
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['font.family'] = 'Lato'

print('matplotlib', matplotlib.__version__)
print('pandas', pd.__version__)
print('numpy', np.__version__)
print('networkx', nx.__version__)

matplotlib 3.8.2
pandas 2.1.3
numpy 1.26.2
networkx 3.4.2


In [2]:
# generate samples and 
def generate_sample_file(varid, ukb_tp53_all, in_samples, out_filename='variant_samples.sample'):

    variant_ids = ukb_tp53_all[ukb_tp53_all.varID == varid]['ID_v0'].drop_duplicates()
    sample_file = pd.read_csv(in_samples, sep=' ', low_memory=False, skiprows = [1]).reset_index().rename(columns={'index':'haplo_sample'})
    variant_samples = sample_file[sample_file.ID_1.isin(variant_ids)].reset_index(drop=True)

    with open(out_filename, 'w') as f:
        f.write('ID_1 ID_2 missing sex\n')
        f.write('0 0 0 D\n')
    variant_samples[['ID_1', 'ID_2', 'missing', 'sex']].to_csv(out_filename, sep=' ', index=False, header=False, mode='a')
    return 0


# wrapper to generate IBD pairs from scratch
# output_dir relative to data_tp53_haplotype/GERMLINE2
# split this into two stages
def extract_haplotypes(varid, scripts_dir, data_dir, tmp_dir, output_dir):

    #data_dir = 'data/haplotype_calling'

    #data_dir = 'data_tp53_haplotype/GERMLINE2'
    #scripts_dir = os.path.expanduser('~/Documents/!Hamish MacGregor/Cambridge/PhD/TP53/data_tp53_haplotype/scripts')
    subset_bgen = scripts_dir+'/subset_bgen.sh'
    vcf_to_hap = scripts_dir+'/vcf_to_hap.py'

    #germline2 = os.path.expanduser('~/germline2/g2')

    # inputs
    in_bgen = data_dir+'/chr17_copy.bgen'
    in_samples = data_dir+'/chr17_imputed_samples.sample'
    #genome_map = data_dir+'/UCSC_hg38_deCODE_recombination_avg_chr17_GERMLINE2.txt'

    # intermediate files
    variant_vcf_root = tmp_dir+'variant_imputed_haps'
    hap_root = tmp_dir+f'/hap_matrices/hap_matrix_{varid}'

    # output
    #output_file = data_dir+f'/{output_dir}/IBD_segments_{varid}.tsv'

    # extract samples that carry variants
    filtered_sample_file = tmp_dir+'/variant_samples.sample'
    #generate_sample_file(varid, ukb_tp53_all, data_dir, filtered_sample_file)


    # qctool to extract haplotypes
    print('qcTool')
    subprocess.run([subset_bgen, in_bgen, in_samples, filtered_sample_file,     # inputs
                    variant_vcf_root],                                          # outputs
                   stdout = subprocess.DEVNULL, stderr=subprocess.DEVNULL)      # remove this line to see printed output

    # hard-call and convert to SHAPEIT format
    print('SHAPEIT')
    subprocess.run(['python', vcf_to_hap, variant_vcf_root+'.vcf',
                    hap_root],
                   stdout = subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    subprocess.run(['rm', filtered_sample_file])

    # germline2
    #print('germline2')
    #subprocess.run([germline2, '-h', hap_root+'.hap', filtered_sample_file, genome_map,
    #                output_file],
    #               stdout = subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    return 0

def germline2_chr17_ibd(varid, germline2, data_dir, tmp_dir, output_dir, min_length=1):

    #data_dir = 'data_tp53_haplotype/GERMLINE2'
    #germline2 = os.path.expanduser('~/germline2/g2')
    genome_map = data_dir+'/UCSC_hg38_deCODE_recombination_avg_chr17_GERMLINE2.txt'

    output_file = output_dir+f'/IBD_segments_{varid}.tsv'

    hap_root = tmp_dir+f'/hap_matrices/hap_matrix_{varid}'
    #hap_matrix = np.loadtxt(hap_root+'.hap', dtype=str)


    subprocess.run([germline2, '-h', f'-m {min_length}', hap_root+'.hap', hap_root+'.sample', genome_map,
                    output_file],
                   stdout = subprocess.DEVNULL, stderr=subprocess.DEVNULL)


def load_qctool_hap_matrix(varid, tmp_dir):
    filename = f'{tmp_dir}/hap_matrices/hap_matrix_{varid}.hap'
    data = []
    positions = []
    with open(filename, 'r') as f:
        for line in f:
            cols = line.strip().split()
            data.append([int(x) for x in cols[5:]])
            positions.append(int(cols[2]))
    return np.array(data), np.array(positions)

In [28]:
# do qctool/vcf conversion for 100 (or more??) sets of 35 samples. 

def generate_random_samples(n, i, data_dir, tmp_dir):

    in_samples = data_dir+'/chr17_imputed_samples.sample'
    sample_file = pd.read_csv(in_samples, sep=' ', low_memory=False, skiprows = [1]).reset_index().rename(columns={'index':'haplo_sample'})
    variant_samples = sample_file.sample(n).reset_index(drop=True)
    out_filename = tmp_dir+f'/variant_random_samples_{i}.sample'

    with open(out_filename, 'w') as f:
        f.write('ID_1 ID_2 missing sex\n')
        f.write('0 0 0 D\n')
    variant_samples[['ID_1', 'ID_2', 'missing', 'sex']].to_csv(out_filename, sep=' ', index=False, header=False, mode='a')
    return


def germline2_chr17_ibd_randoms_stage1(i, scripts_dir, data_dir, tmp_dir, print_per=1):

    
    subset_bgen = scripts_dir+'/subset_bgen.sh'
    vcf_to_hap = scripts_dir+'/vcf_to_hap.py'

    # inputs
    in_bgen = data_dir+'/chr17_copy.bgen'
    in_samples = data_dir+'/chr17_imputed_samples.sample'

    # intermediate files
    variant_vcf_root = data_dir+'variant_imputed_haps'
    hap_root = tmp_dir+f'/hap_matrix_random_{i}' #temp file that is subsequently loaded into a dictionary, which is saved

    # extract samples that carry variants
    filtered_sample_file = tmp_dir+f'/variant_random_samples_{i}.sample'

    # --------------------------------------------------------------------------

    # qctool to extract haplotypes
    if i%print_per == 0: print(i, 'qctool')
    subprocess.run([subset_bgen, in_bgen, in_samples, filtered_sample_file,     # inputs
                    variant_vcf_root],                                          # outputs
                   stdout = subprocess.DEVNULL, stderr=subprocess.DEVNULL)      # remove this line to see printed output

    # hard-call and convert to SHAPEIT format
    subprocess.run(['python', vcf_to_hap, variant_vcf_root+'.vcf',
                    hap_root],
                   stdout = subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    # remove original filtered sample file. this is to avoid confusion - the newer 'hap-matrix' sample file is in the right order (i.e. the order in the hap matrix.hap file). The older sample file is not, it is in the order originally generated. Confused? unsurprising - that's why I'm removing this file so you can't make a mistake. 
    subprocess.run(['rm', filtered_sample_file])
    return 0

def germline2_chr17_ibd_randoms_stage2(i, data_dir, tmp_dir, results_dir, subset_size, min_length=1):
    # takes a subset n samples from the hap file (n should be 35 or less in this case

    #data_dir = 'data_tp53_haplotype/GERMLINE2'
    germline2 = os.path.expanduser('~/germline2/g2')
    # inputs
    genome_map = data_dir+'/UCSC_hg38_deCODE_recombination_avg_chr17_GERMLINE2.txt'

    # intermediate files
    hap_root = tmp_dir+f'/hap_matrix_random_{i}'

    # Create temporary subset files
    hap_subset = tmp_dir+f'/hap_matrix_random_{i}_subset.hap'
    sample_subset = tmp_dir+f'/hap_matrix_random_{i}_subset.sample'

    # output
    output_file = results_dir+f'/IBD_segments_random_{i}.tsv'

    # ---subset sample and hap files ----------------------------------------------------------
    samples_df = pd.read_csv(hap_root+'.sample', sep=' ')
    header_row = samples_df.iloc[0]  # Keep the header row
    data_rows = samples_df.iloc[1:]  # Actual sample data

    # Randomly select n samples
    selected_samples = data_rows.sample(n=subset_size, replace=False)
    selected_indices = selected_samples.index - 1  # Adjust for 0-based indexing (excluding header)

    # Write subset sample file
    subset_samples_df = pd.concat([pd.DataFrame([header_row]), selected_samples])
    subset_samples_df.to_csv(sample_subset, sep=' ', index=False)

    # Read and subset the .hap file
    hap_matrix = np.loadtxt(hap_root+'.hap', dtype=str)

    # Each sample has 2 haplotypes (columns 5 onwards, 2 consecutive columns per sample)
    # Select the haplotype columns corresponding to selected samples
    hap_cols_to_keep = []
    for idx in selected_indices:
        hap_cols_to_keep.extend([5 + 2*idx, 5 + 2*idx + 1])

    # Keep first 5 columns (variant info) plus selected haplotype columns
    all_cols_to_keep = list(range(5)) + hap_cols_to_keep
    hap_subset_matrix = hap_matrix[:, all_cols_to_keep]

    # Write subset hap file
    np.savetxt(hap_subset, hap_subset_matrix, fmt='%s')


    # germline2
    #if i%100 ==0: print(i)
    subprocess.run([germline2, '-h', f'-m {min_length}', hap_subset, sample_subset, genome_map,
                    output_file],
                   stdout = subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    # Clean up temporary files
    os.remove(hap_subset)
    os.remove(sample_subset)

    return 0

def load_qctool_hap_matrix_randoms(hap_root):
    filename = f'data_tp53_haplotype/GERMLINE2/hap_matrices/hap_matrix_{varid}.hap'
    data = []
    positions = []
    with open(filename, 'r') as f:
        for line in f:
            cols = line.strip().split()
            data.append([int(x) for x in cols[5:]])
            positions.append(int(cols[2]))
    return np.array(data), np.array(positions)

def get_g2_segments_random(i, variant_pos, results_dir):

    output_file = results_dir+f'/IBD_segments_random_{i}.tsv'

    g2_segments = pd.read_csv(output_file, sep='\t', header=None, names=['ID_1', 'ID_2', 'P0', 'P1', 'cM', 'words', 'gaps'])

    g2_overlapping_segments = g2_segments[(g2_segments.P0  < variant_pos) & (g2_segments.P1 > variant_pos)].reset_index(drop=True)
    return g2_overlapping_segments

Generate random samples. This takes several hours

In [21]:
# with 35 samples, this takes about one minute per sample. I am going to do 400
n = 35
random.seed(0)
data_dir = 'data/haplotype_calling' 
tmp_dir = 'tmp/random_haplotypes'
scripts_dir = 'scripts'

Path(tmp_dir).mkdir(parents=True, exist_ok=True)
Path(data_dir).mkdir(parents=True, exist_ok=True)


for i in range(400):
    generate_random_samples(n, i, data_dir, hap_dir)
    germline2_chr17_ibd_randoms_stage1(i, scripts_dir, data_dir, tmp_dir, print_per=1)

0 qctool
1 qctool
2 qctool


KeyboardInterrupt: 

In [29]:
# create df for each run
# want to do this again later with a smaller minimum length to allow more sensitivity. 

data_dir = 'data/haplotype_calling'
tmp_dir = 'tmp/random_haplotypes'
results_dir = 'results/random_IBD_segments'

Path(results_dir).mkdir(parents=True, exist_ok=True)

minimum_cM_length = 0.3
for subset in range(2, 35):
    print(subset)

    # Collect all segments in a list
    all_segments = []

    for i in range(400):
        variant_pos = 7675070 # this is the position of TP53 R181H
        germline2_chr17_ibd_randoms_stage2(i, data_dir, tmp_dir, results_dir, subset_size=subset, min_length=minimum_cM_length)
        g2_overlapping_segments = get_g2_segments_random(i, variant_pos, results_dir)

        # append to list if there are any segments for this iteration
        if len(g2_overlapping_segments) > 0:
            g2_overlapping_segments['iteration'] = i
            all_segments.append(g2_overlapping_segments)

        # clean up to avoid thousands of files
        if os.path.exists(f'{tmp_dir}/IBD_segments_random_{i}.tsv'):
            os.remove(f'{tmp_dir}/IBD_segments_random_{i}.tsv')

    # Concatenate all segments
    if all_segments:
        g2_random_segments = pd.concat(all_segments, axis=0, ignore_index=True)
        g2_random_segments.to_csv(f'{results_dir}/overlapping_segments_size_{subset}.tsv', sep='\t', index=False)
    else:
        # Handle case where no segments found in any iteration
        print(f"Warning: No overlapping segments found for subset size {subset}")

2


FileNotFoundError: [Errno 2] No such file or directory: 'tmp/random_haplotypes/hap_matrix_random_5.sample'

In [30]:
frac_with_IBD = {}
for subset in range(2, 35):
    overlapping_segments = pd.read_csv(f'{results_dir}/overlapping_segments_size_{subset}.tsv', sep='\t')
    frac_with_IBD[subset] = []
    block_lengths = np.linspace(0.3, 4, 300)
    for minimum_cM_length in block_lengths:
        overlapping_min_len = overlapping_segments[overlapping_segments.cM > minimum_cM_length].reset_index(drop=True)
        # loop through each iteration and find the number of individuals with IBD 
        number_of_segments_iter = []
        for iteration in range(400):
            overlapping_segments_iter = overlapping_min_len[overlapping_min_len.iteration == iteration]
            ID1 = [x[0] for x in overlapping_segments_iter['ID_1'].astype(str).str.split('.')]
            ID2 = [x[0] for x in overlapping_segments_iter['ID_2'].astype(str).str.split('.')]
            unique_IDs = set(ID1+ID2)
            number_of_segments_iter.append(len(unique_IDs))

        frac_with_IBD[subset].append(np.mean(number_of_segments_iter)/subset)
        

FileNotFoundError: [Errno 2] No such file or directory: 'results/random_IBD_segments/overlapping_segments_size_2.tsv'

In [ ]:
ukb_real_group_sizes = analysis_variants.value_counts('n_ukb').reset_index().sort_values('n_ukb').reset_index(drop=True)
total_indivs = sum(ukb_real_group_sizes['n_ukb']*ukb_real_group_sizes['count'])
false_positives = np.zeros(len(block_lengths))
for i, n in enumerate(ukb_real_group_sizes['n_ukb']):
    multiplier = ukb_real_group_sizes['count'][i]
    mean_false_pos_n = frac_with_IBD[n]

    false_positives += np.array(mean_false_pos_n)*n*multiplier

false_positives_rate = false_positives/total_indivs

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=[15, 5])
ax[0].plot(block_lengths, false_positives_rate, color='k')
#ax.set_ylim(0, 0.10)
ax[0].set_ylim(0, 0.3)
ax[0].set_xlim(0, 4)
ax[0].set_ylabel('fraction of random individuals\nsharing haplotype segment')
ax[0].set_xlabel('minimum shared haplotype length (cM)')
ax[0].vlines(1.5, 0, 0.3, ls='dashed', color='k', alpha=0.5)
plt.tight_layout()


# expected false positive rate by group size
fp_rate_by_n = []
minimum_cM_length = 1.5
for n in range(2, 35):
    fp_rate_by_n.append(np.interp(minimum_cM_length, block_lengths, frac_with_IBD[n]))

ax[1].plot(list(range(2, 35)), fp_rate_by_n, color='k')
ax[1].set_xlim(0, 35)
ax[1].set_ylim(0, 0.05)
ax[1].set_ylabel('fraction of random indiviudals\nsharing haplotype segment')
ax[1].set_xlabel('number of individuals compared')
ax[1].text(10, 0.048, 'Minimum shared haplotype length = 1.5cM', ha='center', va='center', size=12)
plt.tight_layout()
#plt.savefig('TP53_manuscript/figures/hap_length_optimisation.pdf')
plt.show()

### Specific calculations

For example, n = 11, l = 3.5cM (for the R158H carrier)

In [17]:
data_dir = 'data_tp53_haplotype/GERMLINE2'
output_dir = 'hap_matrices_random'

frac_with_IBD = {}
for subset in [11]:
    overlapping_segments = pd.read_csv(f'{data_dir}/{output_dir}/overlapping_segments_size_{subset}.tsv', sep='\t')
    frac_with_IBD[subset] = []
    block_lengths = np.linspace(0.3, 4, 300)
    for minimum_cM_length in block_lengths:
        overlapping_min_len = overlapping_segments[overlapping_segments.cM > minimum_cM_length].reset_index(drop=True)
        # loop through each iteration and find the number of individuals with IBD 
        number_of_segments_iter = []
        for iteration in range(400):
            overlapping_segments_iter = overlapping_min_len[overlapping_min_len.iteration == iteration]
            ID1 = [x[0] for x in overlapping_segments_iter['ID_1'].astype(str).str.split('.')]
            ID2 = [x[0] for x in overlapping_segments_iter['ID_2'].astype(str).str.split('.')]
            unique_IDs = set(ID1+ID2)
            number_of_segments_iter.append(len(unique_IDs))

        frac_with_IBD[subset].append(np.mean(number_of_segments_iter)/subset)
        